# Notebook 04 — Enhanced 3D ResUNet Model Architecture

This notebook defines, builds, validates, and persistently saves the Enhanced 3D ResUNet multi-task model for:

1. Binary rib-fracture segmentation
2. Five-class fracture patch classification

Input: `64 × 64 × 64 × 1` CT patch.

Classification classes:

| Class | Meaning |
|---:|---|
| 0 | Background / negative patch |
| 1 | Displaced fracture |
| 2 | Non-displaced fracture |
| 3 | Buckle fracture |
| 4 | Segmental fracture |

### Notebook separation
- Notebook 03: patch extraction and labels
- Notebook 04: architecture, validation, and persistent model artifacts
- Notebook 05: training
- Notebook 06: final evaluation

Notebook 04 does **not** train the model. Successful execution establishes architectural and computational validity.

### Persistent output
The notebook creates:

```text
/kaggle/working/model_artifacts/
├── enhanced_3d_resunet_multitask.keras
├── model_config.json
└── dataset-metadata.json
```

The directory is intended to be published as the persistent Kaggle Dataset:

`waboke/ribfrac-model-artifacts`

In [4]:
# CELL 1 — IMPORTS AND REPRODUCIBILITY

import os
import json
import random
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Seed:", SEED)

TensorFlow: 2.20.0
Seed: 42


In [5]:
# CELL 2 — GPU CONFIGURATION

gpus = tf.config.list_physical_devices("GPU")
print("Number of GPUs:", len(gpus))

if gpus:
    for gpu in gpus:
        print(gpu)
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth enabled.")
    except RuntimeError as error:
        print("GPU configuration warning:", error)
else:
    print("WARNING: No GPU detected.")

Number of GPUs: 2
PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')
GPU memory growth enabled.


## 1. Persistent input paths

The code supports both the published patch dataset and the current development path.

In [6]:

# ============================================================
# CELL 3 — LOCATE RIBFRAC PATCH DATASET
# Compatible with Notebook 03
# ============================================================

from pathlib import Path
import json

PATCH_DATASET_CANDIDATES = [
    Path("/kaggle/input/datasets/waboke/ribfrac-patches-pilot"),
    Path("/kaggle/input/ribfrac-patches-pilot"),
    Path("/kaggle/input/datasets/waboke/ribfrac-patch-dataset"),
    Path("/kaggle/input/ribfrac-patch-dataset"),
    Path("/kaggle/working/ribfrac-patch-dataset-final"),
    Path("/kaggle/working/patch_dataset"),
]

PATCH_ROOT = None

for candidate in PATCH_DATASET_CANDIDATES:

    roots = [
        candidate,
        candidate / "patch_dataset",
        candidate / "ribfrac-patch-dataset-final",
    ]

    for root in roots:

        if (root / "train").is_dir():
            if any((root / "train").glob("*/images.npy")):
                PATCH_ROOT = root
                break

    if PATCH_ROOT is not None:
        break

if PATCH_ROOT is None:
    raise FileNotFoundError(
        "No RibFrac patch dataset found. "
        "Attach the Notebook 03 patch dataset to Kaggle."
    )

# ------------------------------------------------------------
# Select a completed training patient
# ------------------------------------------------------------

TRAIN_ROOT = PATCH_ROOT / "train"

required_names = [
    "images.npy",
    "segmentation.npy",
    "classification.npy",
    "patch_manifest.csv",
]

completed_training_patients = []

for patient_dir in sorted(TRAIN_ROOT.iterdir()):

    if not patient_dir.is_dir():
        continue

    status_path = (
        patient_dir / "patient_patch_status.json"
    )

    if not status_path.exists():
        continue

    with open(status_path, "r") as f:
        status = json.load(f)

    if status.get("status") != "completed":
        continue

    if all(
        (patient_dir / name).is_file()
        for name in required_names
    ):
        completed_training_patients.append(patient_dir)

if not completed_training_patients:
    raise RuntimeError(
        "No completed training patient found."
    )

# Choose first available completed training patient
DEV_PATIENT_ROOT = completed_training_patients[0]
DEV_PATIENT = DEV_PATIENT_ROOT.name

# Maintain the variable names required by Cells 4–6
X_PATH = DEV_PATIENT_ROOT / "images.npy"
Y_SEG_PATH = DEV_PATIENT_ROOT / "segmentation.npy"
Y_CLASS_PATH = DEV_PATIENT_ROOT / "classification.npy"

print("=" * 65)
print("RIBFRAC PATCH DATASET LOCATION")
print("=" * 65)

print("Patch dataset root:", PATCH_ROOT)
print("Completed training patients:",
      len(completed_training_patients))
print("Selected development patient:", DEV_PATIENT)

print("\nCT patches:", X_PATH)
print("Segmentation masks:", Y_SEG_PATH)
print("Classification labels:", Y_CLASS_PATH)

print("\nCell 3: PASS")


RIBFRAC PATCH DATASET LOCATION
Patch dataset root: /kaggle/input/datasets/waboke/ribfrac-patches-pilot
Completed training patients: 5
Selected development patient: RibFrac10

CT patches: /kaggle/input/datasets/waboke/ribfrac-patches-pilot/train/RibFrac10/images.npy
Segmentation masks: /kaggle/input/datasets/waboke/ribfrac-patches-pilot/train/RibFrac10/segmentation.npy
Classification labels: /kaggle/input/datasets/waboke/ribfrac-patches-pilot/train/RibFrac10/classification.npy

Cell 3: PASS


In [7]:

# ============================================================
# CELL 4 — VERIFY NOTEBOOK 03 OUTPUTS
# ============================================================

required_patch_files = [
    X_PATH,
    Y_SEG_PATH,
    Y_CLASS_PATH
]

for path in required_patch_files:
    assert path.is_file(), f"Missing required file: {path}"
    assert path.stat().st_size > 0, f"Empty file: {path}"

print("=" * 65)
print("NOTEBOOK 03 OUTPUT VERIFICATION")
print("=" * 65)

print("Development patient:", DEV_PATIENT)

for path in required_patch_files:
    print(f"PASS: {path.name}")

print("\nNotebook 03 output validation: PASS")
print("Cell 4: PASS")


NOTEBOOK 03 OUTPUT VERIFICATION
Development patient: RibFrac10
PASS: images.npy
PASS: segmentation.npy
PASS: classification.npy

Notebook 03 output validation: PASS
Cell 4: PASS


In [8]:

# ============================================================
# CELL 5 — LOAD DEVELOPMENT PATCH ARRAYS
# ============================================================

X_dev = np.load(X_PATH, mmap_mode="r")
Y_seg_dev = np.load(Y_SEG_PATH, mmap_mode="r")
Y_class_dev = np.load(Y_CLASS_PATH, mmap_mode="r")

print("X shape:", X_dev.shape)
print("Y_seg shape:", Y_seg_dev.shape)
print("Y_class shape:", Y_class_dev.shape)

print("\nData types:")
print("X dtype:", X_dev.dtype)
print("Y_seg dtype:", Y_seg_dev.dtype)
print("Y_class dtype:", Y_class_dev.dtype)

print("\nCell 5: PASS")


X shape: (100, 64, 64, 64, 1)
Y_seg shape: (100, 64, 64, 64, 1)
Y_class shape: (100,)

Data types:
X dtype: float32
Y_seg dtype: uint8
Y_class dtype: int8

Cell 5: PASS


In [9]:

# ============================================================
# CELL 6 — VALIDATE PATCH DATA
# ============================================================

# Expected 3D CNN input: (N, 64, 64, 64, 1)
assert X_dev.ndim == 5, f"Unexpected X_dev shape: {X_dev.shape}"
assert X_dev.shape[1:] == (64, 64, 64, 1)

# Expected segmentation target: (N, 64, 64, 64, 1)
assert Y_seg_dev.ndim == 5, f"Unexpected Y_seg_dev shape: {Y_seg_dev.shape}"
assert Y_seg_dev.shape[1:] == (64, 64, 64, 1)

# Expected classification target: (N,)
assert Y_class_dev.ndim == 1, (
    f"Unexpected Y_class_dev shape: {Y_class_dev.shape}"
)

# All arrays must contain the same number of patches
assert X_dev.shape[0] == Y_seg_dev.shape[0] == Y_class_dev.shape[0]

# Check values
assert np.isfinite(X_dev).all(), "CT data contains NaN or infinity."

assert set(np.unique(Y_seg_dev)).issubset({0, 1}), (
    "Segmentation targets contain values outside {0, 1}."
)

assert set(np.unique(Y_class_dev)).issubset({0, 1, 2, 3, 4}), (
    "Classification targets contain unexpected class labels."
)

print("Patch data validation passed.")
print("X_dev:", X_dev.shape, X_dev.dtype)
print("Y_seg_dev:", Y_seg_dev.shape, Y_seg_dev.dtype)
print("Y_class_dev:", Y_class_dev.shape, Y_class_dev.dtype)
print("Segmentation values:", np.unique(Y_seg_dev))
print("Classification classes:", np.unique(Y_class_dev))


Patch data validation passed.
X_dev: (100, 64, 64, 64, 1) float32
Y_seg_dev: (100, 64, 64, 64, 1) uint8
Y_class_dev: (100,) int8
Segmentation values: [0 1]
Classification classes: [0 1 2 3]


## 2. Model configuration

Baseline architecture:

- Input: `64 × 64 × 64 × 1`
- Encoder depth: 4
- Base filters: 16
- Encoder filters: 16, 32, 64, 128
- Bottleneck: 256
- Residual 3D convolution blocks
- U-Net skip connections
- 3D transposed-convolution upsampling
- Segmentation: sigmoid
- Classification: five-class softmax

In [10]:

# ============================================================
# CELL 7 — MODEL CONFIGURATION
# ============================================================

INPUT_SHAPE = (64, 64, 64, 1)
PATCH_SIZE = (64, 64, 64)

NUM_CLASSES = 5

CLASS_NAMES = {
    0: "background",
    1: "displaced",
    2: "non-displaced",
    3: "buckle",
    4: "segmental",
}

BASE_FILTERS = 16
DEPTH = 4
DROPOUT_RATE = 0.20

LEARNING_RATE = 1e-4
SEGMENTATION_LOSS_WEIGHT = 1.0
CLASSIFICATION_LOSS_WEIGHT = 1.0

MODEL_NAME = "Enhanced_3D_ResUNet_MultiTask"

print("=" * 65)
print("3D RESUNET — MODEL CONFIGURATION")
print("=" * 65)

print("Input shape:", INPUT_SHAPE)
print("Number of classes:", NUM_CLASSES)
print("Class names:", CLASS_NAMES)
print("Base filters:", BASE_FILTERS)
print("Depth:", DEPTH)
print("Dropout:", DROPOUT_RATE)
print("Learning rate:", LEARNING_RATE)
print("Segmentation loss weight:", SEGMENTATION_LOSS_WEIGHT)
print("Classification loss weight:", CLASSIFICATION_LOSS_WEIGHT)
print("Model name:", MODEL_NAME)

print("\nCell 7: PASS")


3D RESUNET — MODEL CONFIGURATION
Input shape: (64, 64, 64, 1)
Number of classes: 5
Class names: {0: 'background', 1: 'displaced', 2: 'non-displaced', 3: 'buckle', 4: 'segmental'}
Base filters: 16
Depth: 4
Dropout: 0.2
Learning rate: 0.0001
Segmentation loss weight: 1.0
Classification loss weight: 1.0
Model name: Enhanced_3D_ResUNet_MultiTask

Cell 7: PASS


## 3. Enhanced residual 3D building blocks

In [11]:
# ============================================================
# CELL 8 — RESIDUAL 3D CONVOLUTION BLOCK
# ============================================================

def residual_block_3d(x, filters, name, dropout_rate=0.0):

    # Save input for shortcut connection
    shortcut = x

    # First 3D convolution
    x = layers.Conv3D(
        filters,
        3,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name=f"{name}_conv1"
    )(x)

    x = layers.BatchNormalization(
        name=f"{name}_bn1"
    )(x)

    x = layers.Activation(
        "relu",
        name=f"{name}_relu1"
    )(x)

    # Second 3D convolution
    x = layers.Conv3D(
        filters,
        3,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name=f"{name}_conv2"
    )(x)

    x = layers.BatchNormalization(
        name=f"{name}_bn2"
    )(x)

    # Match shortcut channels if necessary
    input_channels = shortcut.shape[-1]

    if input_channels != filters:

        shortcut = layers.Conv3D(
            filters,
            1,
            padding="same",
            use_bias=False,
            kernel_initializer="he_normal",
            name=f"{name}_projection"
        )(shortcut)

        shortcut = layers.BatchNormalization(
            name=f"{name}_projection_bn"
        )(shortcut)

    # Residual addition
    x = layers.Add(
        name=f"{name}_add"
    )([x, shortcut])

    x = layers.Activation(
        "relu",
        name=f"{name}_out"
    )(x)

    # Optional dropout
    if dropout_rate > 0:

        x = layers.Dropout(
            dropout_rate,
            name=f"{name}_dropout"
        )(x)

    return x


print("Residual 3D convolution block defined successfully.")
print("Cell 8: PASS")


Residual 3D convolution block defined successfully.
Cell 8: PASS


In [12]:

# ============================================================
# CELL 9 — ENCODER DOWNSAMPLING BLOCK
# ============================================================

def downsample_block_3d(x, filters, name):
    return layers.Conv3D(
        filters,
        kernel_size=2,
        strides=2,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name=name,
    )(x)

print("Encoder downsampling block defined successfully.")
print("Cell 9: PASS")


Encoder downsampling block defined successfully.
Cell 9: PASS


In [13]:

# ============================================================
# CELL 10 — DECODER BLOCK
# ============================================================

def decoder_block_3d(
    x, skip, filters, name, dropout_rate=0.0
):
    x = layers.Conv3DTranspose(
        filters,
        kernel_size=2,
        strides=2,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name=f"{name}_up",
    )(x)

    x = layers.Concatenate(
        name=f"{name}_concat"
    )([x, skip])

    x = residual_block_3d(
        x,
        filters,
        name=f"{name}_res1",
        dropout_rate=dropout_rate,
    )

    return x


print("Decoder block defined successfully.")
print("Cell 10: PASS")


Decoder block defined successfully.
Cell 10: PASS


## 4. Build the complete multi-task Enhanced 3D ResUNet

The model has a shared residual 3D encoder and two task-specific branches.

**Segmentation branch:** U-Net decoder with residual blocks and skip connections.

**Classification branch:** global feature aggregation followed by dense layers and a five-class softmax output.

In [14]:

# ============================================================
# CELL 11 — BUILD ENHANCED 3D RESUNET
# ============================================================

def build_enhanced_3d_resunet(
    input_shape=INPUT_SHAPE,
    num_classes=NUM_CLASSES,
    base_filters=BASE_FILTERS,
    dropout_rate=DROPOUT_RATE,
):
    inputs = keras.Input(
        shape=input_shape,
        name="ct_input",
    )

    # --------------------------------------------------------
    # Encoder
    # --------------------------------------------------------

    e1 = residual_block_3d(
        inputs, base_filters, "enc1", dropout_rate
    )
    d1 = downsample_block_3d(
        e1, base_filters * 2, "down1"
    )

    e2 = residual_block_3d(
        d1, base_filters * 2, "enc2", dropout_rate
    )
    d2 = downsample_block_3d(
        e2, base_filters * 4, "down2"
    )

    e3 = residual_block_3d(
        d2, base_filters * 4, "enc3", dropout_rate
    )
    d3 = downsample_block_3d(
        e3, base_filters * 8, "down3"
    )

    e4 = residual_block_3d(
        d3, base_filters * 8, "enc4", dropout_rate
    )
    d4 = downsample_block_3d(
        e4, base_filters * 16, "down4"
    )

    # --------------------------------------------------------
    # Residual bottleneck
    # --------------------------------------------------------

    bottleneck = residual_block_3d(
        d4, base_filters * 16, "bottleneck", dropout_rate
    )

    # --------------------------------------------------------
    # Segmentation decoder
    # --------------------------------------------------------

    dec4 = decoder_block_3d(
        bottleneck, e4, base_filters * 8, "dec4", dropout_rate
    )

    dec3 = decoder_block_3d(
        dec4, e3, base_filters * 4, "dec3", dropout_rate
    )

    dec2 = decoder_block_3d(
        dec3, e2, base_filters * 2, "dec2", dropout_rate
    )

    dec1 = decoder_block_3d(
        dec2, e1, base_filters, "dec1", dropout_rate
    )

    segmentation_output = layers.Conv3D(
        1,
        kernel_size=1,
        activation="sigmoid",
        name="segmentation_output",
    )(dec1)

    # --------------------------------------------------------
    # Classification branch
    # --------------------------------------------------------

    cls = layers.GlobalAveragePooling3D(
        name="classification_global_pool"
    )(bottleneck)

    cls = layers.Dense(
        base_filters * 8,
        activation="relu",
        kernel_initializer="he_normal",
        name="classification_dense1",
    )(cls)

    cls = layers.Dropout(
        dropout_rate,
        name="classification_dropout",
    )(cls)

    cls = layers.Dense(
        base_filters * 4,
        activation="relu",
        kernel_initializer="he_normal",
        name="classification_dense2",
    )(cls)

    classification_output = layers.Dense(
        num_classes,
        activation="softmax",
        name="classification_output",
    )(cls)

    # --------------------------------------------------------
    # Complete multi-task model
    # --------------------------------------------------------

    return keras.Model(
        inputs=inputs,
        outputs={
            "segmentation_output": segmentation_output,
            "classification_output": classification_output,
        },
        name=MODEL_NAME,
    )


print("Enhanced 3D ResUNet architecture defined successfully.")
print("Cell 11: PASS")


Enhanced 3D ResUNet architecture defined successfully.
Cell 11: PASS


In [15]:

# ============================================================
# CELL 12 — INSTANTIATE MODEL
# ============================================================

model = build_enhanced_3d_resunet()

model.summary()

print("Model construction: PASS")


I0000 00:00:1793969986.603088      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1793969986.606536      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "Enhanced_3D_ResUNet_MultiTask"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ ct_input            │ (None, 64, 64,    │          0 │ -                 │
│ (InputLayer)        │ 64, 1)            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc1_conv1 (Conv3D) │ (None, 64, 64,    │        432 │ ct_input[0][0]    │
│                     │ 64, 16)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc1_bn1            │ (None, 64, 64,    │         64 │ enc1_conv1[0][0]  │
│ (BatchNormalizatio… │ 64, 16)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc1_relu1          │ (None, 64, 64,    │          0 │ enc1_bn1[0][0]    │
│ (Activation)        │ 64, 16)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc1_conv2 (Conv3D) │ (None, 64, 64,    │      6,912 │ enc1_relu1[0][0]  │
│                     │ 64, 16)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc1_projection     │ (None, 64, 64,    │         16 │ ct_input[0][0]    │
│ (Conv3D)            │ 64, 16)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc1_bn2            │ (None, 64, 64,    │         64 │ enc1_conv2[0][0]  │
│ (BatchNormalizatio… │ 64, 16)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc1_projection_bn  │ (None, 64, 64,    │         64 │ enc1_projection[… │
│ (BatchNormalizatio… │ 64, 16)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc1_add (Add)      │ (None, 64, 64,    │          0 │ enc1_bn2[0][0],   │
│                     │ 64, 16)           │            │ enc1_projection_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc1_out            │ (None, 64, 64,    │          0 │ enc1_add[0][0]    │
│ (Activation)        │ 64, 16)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc1_dropout        │ (None, 64, 64,    │          0 │ enc1_out[0][0]    │
│ (Dropout)           │ 64, 16)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ down1 (Conv3D)      │ (None, 32, 32,    │      4,096 │ enc1_dropout[0][… │
│                     │ 32, 32)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc2_conv1 (Conv3D) │ (None, 32, 32,    │     27,648 │ down1[0][0]       │
│                     │ 32, 32)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc2_bn1            │ (None, 32, 32,    │        128 │ enc2_conv1[0][0]  │
│ (BatchNormalizatio… │ 32, 32)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc2_relu1          │ (None, 32, 32,    │          0 │ enc2_bn1[0][0]    │
│ (Activation)        │ 32, 32)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc2_conv2 (Conv3D) │ (None, 32, 32,    │     27,648 │ enc2_relu1[0][0]  │
│                     │ 32, 32)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc2_bn2            │ (None, 32, 32,    │        128 │ enc2_conv2[0][0]

 Total params: 7,258,326 (27.69 MB)

 Trainable params: 7,254,870 (27.68 MB)

 Non-trainable params: 3,456 (13.50 KB)

Model construction: PASS


In [16]:

# ============================================================
# CELL 13 — VALIDATE MODEL OUTPUT SHAPES
# ============================================================

print("=" * 65)
print("3D RESUNET — MODEL OUTPUT VALIDATION")
print("=" * 65)

print("Model name:", model.name)
print("Model output shapes:", model.output_shape)
print("Output names:", model.output_names)

# ------------------------------------------------------------
# 1. Retrieve output shapes by layer name
# ------------------------------------------------------------

segmentation_shape = tuple(
    model.get_layer("segmentation_output").output.shape
)

classification_shape = tuple(
    model.get_layer("classification_output").output.shape
)

# ------------------------------------------------------------
# 2. Validate segmentation output
# ------------------------------------------------------------

expected_segmentation_shape = (
    None, 64, 64, 64, 1
)

assert segmentation_shape == expected_segmentation_shape, (
    f"Unexpected segmentation output: {segmentation_shape}"
)

# ------------------------------------------------------------
# 3. Validate classification output
# ------------------------------------------------------------

expected_classification_shape = (
    None, NUM_CLASSES
)

assert classification_shape == expected_classification_shape, (
    f"Unexpected classification output: {classification_shape}"
)

# ------------------------------------------------------------
# 4. Validate both output branches
# ------------------------------------------------------------

assert set(model.output_names) == {
    "segmentation_output",
    "classification_output",
}, "Missing or unexpected model output branch."

print("\nSegmentation output:", segmentation_shape)
print("Classification output:", classification_shape)

print("\nModel output validation: PASS")
print("Cell 13: PASS")


3D RESUNET — MODEL OUTPUT VALIDATION
Model name: Enhanced_3D_ResUNet_MultiTask
Model output shapes: [(None, 5), (None, 64, 64, 64, 1)]
Output names: ListWrapper(['classification_output', 'segmentation_output'])

Segmentation output: (None, 64, 64, 64, 1)
Classification output: (None, 5)

Model output validation: PASS
Cell 13: PASS


In [17]:

# ============================================================
# CELL 14 — FORWARD PASS TEST
# Enhanced 3D ResUNet
# ============================================================

# Start with one patch to minimize GPU memory usage
test_batch_size = 1

# Load CT patch with existing channel dimension
x_test = np.asarray(
    X_dev[:test_batch_size],
    dtype=np.float32
)

print("=" * 65)
print("3D RESUNET — FORWARD PASS TEST")
print("=" * 65)

print("Input shape:", x_test.shape)

assert x_test.shape == (
    test_batch_size, 64, 64, 64, 1
)

# ------------------------------------------------------------
# 1. Forward propagation
# ------------------------------------------------------------

outputs = model(
    x_test,
    training=False
)

# ------------------------------------------------------------
# 2. Retrieve model predictions
# ------------------------------------------------------------

seg_pred = outputs["segmentation_output"].numpy()
cls_pred = outputs["classification_output"].numpy()

# ------------------------------------------------------------
# 3. Validate prediction shapes
# ------------------------------------------------------------

assert seg_pred.shape == (
    test_batch_size, 64, 64, 64, 1
)

assert cls_pred.shape == (
    test_batch_size, NUM_CLASSES
)

# ------------------------------------------------------------
# 4. Validate segmentation predictions
# ------------------------------------------------------------

assert np.isfinite(seg_pred).all()

assert np.all(seg_pred >= 0.0)
assert np.all(seg_pred <= 1.0)

# ------------------------------------------------------------
# 5. Validate classification probabilities
# ------------------------------------------------------------

assert np.isfinite(cls_pred).all()

cls_sums = cls_pred.sum(axis=1)

assert np.allclose(
    cls_sums,
    1.0,
    atol=1e-5
)

# ------------------------------------------------------------
# 6. Display results
# ------------------------------------------------------------

print("\nSegmentation prediction shape:", seg_pred.shape)
print("Classification prediction shape:", cls_pred.shape)

print("\nSegmentation probability range:")
print("Minimum:", float(seg_pred.min()))
print("Maximum:", float(seg_pred.max()))

print("\nClassification probabilities:")
print(cls_pred)

print("\nClassification probability sums:")
print(cls_sums)

print("\nForward pass: PASS")
print("Cell 14: PASS")


3D RESUNET — FORWARD PASS TEST
Input shape: (1, 64, 64, 64, 1)

Segmentation prediction shape: (1, 64, 64, 64, 1)
Classification prediction shape: (1, 5)

Segmentation probability range:
Minimum: 9.516440258039438e-08
Maximum: 0.9989256262779236

Classification probabilities:
[[0.22962517 0.16097708 0.28523138 0.28154442 0.04262192]]

Classification probability sums:
[0.99999994]

Forward pass: PASS
Cell 14: PASS


## 5. Compile the multi-task model

Baseline losses:

- Segmentation: Binary Cross-Entropy
- Classification: Sparse Categorical Cross-Entropy

Baseline loss weights:

- Segmentation = 1.0
- Classification = 1.0

Compilation here is only for validation. Actual optimization, callbacks, augmentation, and training belong to Notebook 05.

In [18]:

# ============================================================
# CELL 15 — BASELINE MULTI-TASK COMPILATION
# ============================================================

model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    ),

    loss={
        "segmentation_output":
            keras.losses.BinaryCrossentropy(),

        "classification_output":
            keras.losses.SparseCategoricalCrossentropy(),
    },

    loss_weights={
        "segmentation_output":
            SEGMENTATION_LOSS_WEIGHT,

        "classification_output":
            CLASSIFICATION_LOSS_WEIGHT,
    },

    metrics={
        "segmentation_output": [
            keras.metrics.BinaryAccuracy(
                name="seg_accuracy"
            )
        ],

        "classification_output": [
            keras.metrics.SparseCategoricalAccuracy(
                name="cls_accuracy"
            )
        ],
    },
)

print("Model compilation: PASS")


Model compilation: PASS


In [19]:

# ============================================================
# CELL 16 — LOSS AND GRADIENT SMOKE TEST
# Enhanced 3D Residual U-Net
# ============================================================

import numpy as np
import tensorflow as tf

# ------------------------------------------------------------
# 1. Prepare one CT patch and corresponding targets
# ------------------------------------------------------------

smoke_batch_size = 1

x_smoke = tf.convert_to_tensor(
    np.asarray(
        X_dev[:smoke_batch_size],
        dtype=np.float32
    )
)

y_seg_smoke = tf.convert_to_tensor(
    np.asarray(
        Y_seg_dev[:smoke_batch_size],
        dtype=np.float32
    )
)

y_cls_smoke = tf.convert_to_tensor(
    np.asarray(
        Y_class_dev[:smoke_batch_size],
        dtype=np.int32
    ).reshape(-1)
)

assert tuple(x_smoke.shape) == (1, 64, 64, 64, 1)
assert tuple(y_seg_smoke.shape) == (1, 64, 64, 64, 1)
assert tuple(y_cls_smoke.shape) == (1,)

print("=" * 65)
print("3D RESUNET — LOSS AND GRADIENT SMOKE TEST")
print("=" * 65)

print("Input shape:", x_smoke.shape)
print("Segmentation target:", y_seg_smoke.shape)
print("Classification target:", y_cls_smoke.numpy())

# ------------------------------------------------------------
# 2. Define baseline loss functions
# ------------------------------------------------------------

seg_loss_fn = keras.losses.BinaryCrossentropy()

cls_loss_fn = (
    keras.losses.SparseCategoricalCrossentropy()
)

# Preserve BatchNorm running statistics during the smoke test
bn_snapshots = [
    (
        layer,
        tf.identity(layer.moving_mean),
        tf.identity(layer.moving_variance)
    )
    for layer in model.layers
    if isinstance(layer, keras.layers.BatchNormalization)
]

try:

    # --------------------------------------------------------
    # 3. Forward pass and loss calculation
    # --------------------------------------------------------

    with tf.GradientTape() as tape:

        predictions = model(
            x_smoke,
            training=True
        )

        seg_loss = seg_loss_fn(
            y_seg_smoke,
            predictions["segmentation_output"]
        )

        cls_loss = cls_loss_fn(
            y_cls_smoke,
            predictions["classification_output"]
        )

        # Include any layer regularization losses
        reg_loss = (
            tf.add_n(model.losses)
            if model.losses
            else tf.constant(0.0, dtype=tf.float32)
        )

        total_loss = (
            SEGMENTATION_LOSS_WEIGHT * seg_loss
            + CLASSIFICATION_LOSS_WEIGHT * cls_loss
            + reg_loss
        )

    # --------------------------------------------------------
    # 4. Backpropagation
    # --------------------------------------------------------

    gradients = tape.gradient(
        total_loss,
        model.trainable_variables
    )

    missing_gradients = [
        variable.name
        for variable, gradient in zip(
            model.trainable_variables,
            gradients
        )
        if gradient is None
    ]

    assert not missing_gradients, (
        f"Missing gradients: {missing_gradients}"
    )

    # --------------------------------------------------------
    # 5. Validate numerical stability
    # --------------------------------------------------------

    assert bool(
        tf.math.is_finite(total_loss).numpy()
    ), "Total loss is not finite."

    assert all(
        bool(tf.reduce_all(tf.math.is_finite(g)).numpy())
        for g in gradients
    ), "Some gradients contain NaN or infinity."

    assert any(
        bool(tf.reduce_any(tf.not_equal(g, 0)).numpy())
        for g in gradients
    ), "All gradients are zero."

    # --------------------------------------------------------
    # 6. Display results
    # --------------------------------------------------------

    print("\nSegmentation loss:",
          float(seg_loss.numpy()))

    print("Classification loss:",
          float(cls_loss.numpy()))

    print("Total loss:",
          float(total_loss.numpy()))

    print("\nTrainable variables:",
          len(model.trainable_variables))

    print("Valid gradient tensors:",
          len(gradients))

    print("Missing gradients:",
          len(missing_gradients))

    print("\nLoss/gradient smoke test: PASS")
    print("Cell 16: PASS")

finally:

    # Restore BatchNorm running statistics:
    # no optimization step is performed in this test
    for layer, mean, variance in bn_snapshots:
        layer.moving_mean.assign(mean)
        layer.moving_variance.assign(variance)


3D RESUNET — LOSS AND GRADIENT SMOKE TEST
Input shape: (1, 64, 64, 64, 1)
Segmentation target: (1, 64, 64, 64, 1)
Classification target: [3]

Segmentation loss: 0.503089427947998
Classification loss: 0.6530203223228455
Total loss: 1.1561098098754883

Trainable variables: 85
Valid gradient tensors: 85
Missing gradients: 0

Loss/gradient smoke test: PASS
Cell 16: PASS


## 6. Save persistent model artifacts

Notebook 05 will load these files from the published Kaggle Dataset rather than depending on Notebook 04's temporary runtime.

In [20]:

# ============================================================
# CELL 17 — CREATE PERSISTENT OUTPUT DIRECTORY
# ============================================================

MODEL_ARTIFACT_ROOT = Path(
    "/kaggle/working/model_artifacts"
)

MODEL_ARTIFACT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_PATH = (
    MODEL_ARTIFACT_ROOT
    / "enhanced_3d_resunet_multitask.keras"
)

CONFIG_PATH = (
    MODEL_ARTIFACT_ROOT
    / "model_config.json"
)

METADATA_PATH = (
    MODEL_ARTIFACT_ROOT
    / "dataset-metadata.json"
)

print("Model artifact directory:",
      MODEL_ARTIFACT_ROOT)

assert MODEL_ARTIFACT_ROOT.is_dir()

print("Cell 17: PASS")


Model artifact directory: /kaggle/working/model_artifacts
Cell 17: PASS


In [21]:

# ============================================================
# CELL 18 — SAVE MODEL
# ============================================================

model.save(MODEL_PATH)

assert MODEL_PATH.exists()
assert MODEL_PATH.stat().st_size > 0

print("Model saved:", MODEL_PATH)
print(
    "File size (MB):",
    round(MODEL_PATH.stat().st_size / (1024**2), 2)
)

print("Cell 18: PASS")


Model saved: /kaggle/working/model_artifacts/enhanced_3d_resunet_multitask.keras
File size (MB): 28.03
Cell 18: PASS


In [22]:

# ============================================================
# CELL 19 — SAVE MODEL CONFIGURATION
# ============================================================

model_config = {
    "model_name": MODEL_NAME,
    "input_shape": list(INPUT_SHAPE),
    "patch_size": list(PATCH_SIZE),
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "base_filters": BASE_FILTERS,
    "depth": DEPTH,
    "dropout_rate": DROPOUT_RATE,

    "segmentation_output": {
        "channels": 1,
        "activation": "sigmoid",
        "task": "binary fracture segmentation",
    },

    "classification_output": {
        "classes": NUM_CLASSES,
        "activation": "softmax",
        "task": "five-class patch classification",
    },

    "optimizer": "Adam",
    "learning_rate": LEARNING_RATE,

    "losses": {
        "segmentation_output":
            "BinaryCrossentropy",
        "classification_output":
            "SparseCategoricalCrossentropy",
    },

    "loss_weights": {
        "segmentation_output":
            SEGMENTATION_LOSS_WEIGHT,
        "classification_output":
            CLASSIFICATION_LOSS_WEIGHT,
    },

    "seed": SEED,
    "source_notebook": "Notebook 04",
    "purpose": (
        "Persistent architecture artifact for Notebook 05"
    ),
}

with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(model_config, f, indent=2)

assert CONFIG_PATH.exists()
assert CONFIG_PATH.stat().st_size > 0

print("Configuration saved:", CONFIG_PATH)
print("Cell 19: PASS")


Configuration saved: /kaggle/working/model_artifacts/model_config.json
Cell 19: PASS


In [23]:

# ============================================================
# CELL 20 — CREATE KAGGLE DATASET METADATA
# ============================================================

import json

dataset_metadata = {
    "title": "RibFrac Enhanced 3D ResUNet Model Artifacts",

    "id": "waboke/ribfrac-model-artifacts",

    "licenses": [
        {"name": "CC-BY-4.0"}
    ],

    "description": (
        "Persistent 3D Residual U-Net (3D ResUNet) "
        "multi-task model artifacts produced by "
        "Notebook 04 for rib-fracture segmentation "
        "and five-class patch classification using "
        "RibFrac chest CT images. The model has "
        "passed architecture, forward-pass, and "
        "gradient validation, but its weights are "
        "not yet trained. Contains the Keras model "
        "and reproducible configuration for "
        "Notebook 05 training."
    ),
}

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(dataset_metadata, f, indent=2)

assert METADATA_PATH.exists()
assert METADATA_PATH.stat().st_size > 0

print("=" * 65)
print("KAGGLE MODEL DATASET METADATA")
print("=" * 65)

print("Dataset title:", dataset_metadata["title"])
print("Dataset ID:", dataset_metadata["id"])
print("Metadata saved:", METADATA_PATH)

print("\nCell 20: PASS")


KAGGLE MODEL DATASET METADATA
Dataset title: RibFrac Enhanced 3D ResUNet Model Artifacts
Dataset ID: waboke/ribfrac-model-artifacts
Metadata saved: /kaggle/working/model_artifacts/dataset-metadata.json

Cell 20: PASS


## 7. Reload and validate the persisted model

This step is important: Notebook 05 must be able to load the saved `.keras` artifact independently.

In [24]:

# ============================================================
# CELL 21 — RELOAD SAVED MODEL
# ============================================================

reloaded_model = keras.models.load_model(
    MODEL_PATH,
    compile=False
)

# Verify input and output shapes
assert reloaded_model.input_shape == model.input_shape
assert reloaded_model.output_shape == model.output_shape

# Forward pass using the CT patch from Cell 14
reloaded_outputs = reloaded_model(
    x_test,
    training=False
)

# Retrieve segmentation predictions
reloaded_seg = (
    reloaded_outputs["segmentation_output"]
    .numpy()
)

# Retrieve classification predictions
reloaded_cls = (
    reloaded_outputs["classification_output"]
    .numpy()
)

# Validate prediction dimensions
assert reloaded_seg.shape == seg_pred.shape
assert reloaded_cls.shape == cls_pred.shape

# Verify classification probabilities
assert np.allclose(
    reloaded_cls.sum(axis=1),
    1.0,
    atol=1e-5
)

print("Reload validation: PASS")
print("Reloaded model:", reloaded_model.name)


Reload validation: PASS
Reloaded model: Enhanced_3D_ResUNet_MultiTask


In [25]:

# Cell 21A — Verify saved model prediction consistency

seg_match = np.allclose(
    seg_pred,
    reloaded_seg,
    rtol=1e-5,
    atol=1e-6
)

cls_match = np.allclose(
    cls_pred,
    reloaded_cls,
    rtol=1e-5,
    atol=1e-6
)

print("Segmentation predictions match:", seg_match)
print("Classification predictions match:", cls_match)

assert seg_match, "Segmentation predictions changed."
assert cls_match, "Classification predictions changed."

print("\nModel save/reload consistency: PASS")


Segmentation predictions match: True
Classification predictions match: True

Model save/reload consistency: PASS


In [26]:

# ============================================================
# CELL 22 — FINAL ARTIFACT VALIDATION
# ============================================================

required_artifacts = [
    MODEL_PATH,
    CONFIG_PATH,
    METADATA_PATH,
]

for path in required_artifacts:
    assert path.exists(), f"Missing artifact: {path}"
    assert path.stat().st_size > 0, f"Empty artifact: {path}"

with open(
    CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:
    saved_config = json.load(f)

assert saved_config["model_name"] == MODEL_NAME
assert saved_config["input_shape"] == list(INPUT_SHAPE)
assert saved_config["patch_size"] == list(PATCH_SIZE)
assert saved_config["num_classes"] == NUM_CLASSES
assert saved_config["base_filters"] == BASE_FILTERS
assert saved_config["depth"] == DEPTH

print("=" * 60)
print("NOTEBOOK 04 COMPLETION: PASS")
print("=" * 60)
print("Model:", MODEL_PATH)
print("Configuration:", CONFIG_PATH)
print("Metadata:", METADATA_PATH)
print()
print("Publish as:")
print("waboke/ribfrac-model-artifacts")
print()
print("Notebook 05 will load:")
print("  enhanced_3d_resunet_multitask.keras")
print("  model_config.json")


NOTEBOOK 04 COMPLETION: PASS
Model: /kaggle/working/model_artifacts/enhanced_3d_resunet_multitask.keras
Configuration: /kaggle/working/model_artifacts/model_config.json
Metadata: /kaggle/working/model_artifacts/dataset-metadata.json

Publish as:
waboke/ribfrac-model-artifacts

Notebook 05 will load:
  enhanced_3d_resunet_multitask.keras
  model_config.json


## 8. Publish the model artifacts as a persistent Kaggle Dataset

After all validation cells pass, publish the directory.

### First-time creation

```python
!kaggle datasets create -p /kaggle/working/model_artifacts
```

### Updating an existing dataset

```python
!kaggle datasets version -p /kaggle/working/model_artifacts -m "Updated Notebook 04 model artifacts"
```

Target dataset:

```text
waboke/ribfrac-model-artifacts
```

If Kaggle reports that the dataset already exists, use the `version` command instead of `create`.

In [29]:
#CELL 23 — OPTIONAL KAGGLE PUBLISH COMMAND

#Run ONE of these when ready:

#First publication:
#!kaggle datasets create -p /kaggle/working/model_artifacts

#Existing dataset update:
!kaggle datasets version -p /kaggle/working/model_artifacts -m "Updated Notebook 04 model artifacts"

print("Publishing is optional.")
print("Use the commands in the preceding markdown cell.")

Starting upload for file enhanced_3d_resunet_multitask.keras
100%|██████████████████████████████████████| 28.0M/28.0M [00:04<00:00, 6.68MB/s]
Upload successful: enhanced_3d_resunet_multitask.keras (28MB)
Starting upload for file model_config.json
100%|██████████████████████████████████████████| 999/999 [00:00<00:00, 1.29kB/s]
Upload successful: model_config.json (999B)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/waboke/ribfrac-model-artifacts
Publishing is optional.
Use the commands in the preceding markdown cell.


In [30]:

# ============================================================
# CELL 23A — VERIFY MODEL ARTIFACTS BEFORE PUBLICATION
# ============================================================

from pathlib import Path

artifact_root = Path("/kaggle/working/model_artifacts")

required_files = [
    "enhanced_3d_resunet_multitask.keras",
    "model_config.json",
    "dataset-metadata.json"
]

print("=" * 65)
print("3D RESUNET — MODEL PUBLICATION CHECK")
print("=" * 65)

for filename in required_files:
    path = artifact_root / filename

    assert path.is_file(), f"Missing file: {filename}"
    assert path.stat().st_size > 0, f"Empty file: {filename}"

    print(
        f"PASS: {filename} "
        f"({path.stat().st_size / 1024**2:.2f} MiB)"
    )

print("\nModel artifacts ready for publication.")
print("Cell 23A: PASS")


3D RESUNET — MODEL PUBLICATION CHECK
PASS: enhanced_3d_resunet_multitask.keras (28.03 MiB)
PASS: model_config.json (0.00 MiB)
PASS: dataset-metadata.json (0.00 MiB)

Model artifacts ready for publication.
Cell 23A: PASS


# Notebook 04 completion criteria

The notebook is complete when:

- Notebook 03 patch data is located.
- Patch shapes and class labels are valid.
- Enhanced 3D ResUNet is constructed.
- Input shape is `64×64×64×1`.
- Segmentation output is `64×64×64×1`.
- Classification output is five-class softmax.
- Forward pass succeeds.
- Classification probabilities sum to one.
- Multi-task losses can be calculated.
- Gradients are finite.
- The model saves successfully.
- `model_config.json` is saved.
- `dataset-metadata.json` is saved.
- The saved model reloads successfully.
- Final artifact validation passes.

### Expected output

```text
model_artifacts/
├── enhanced_3d_resunet_multitask.keras
├── model_config.json
└── dataset-metadata.json
```

After publishing this directory as `waboke/ribfrac-model-artifacts`, Notebook 05 can load the exact validated architecture independently of Notebook 04.

**Important:** successful execution of Notebook 04 demonstrates architectural and computational validity; it does not establish predictive performance. Training belongs to Notebook 05 and final evaluation belongs to Notebook 06.